# Geocoding, 날씨 API 활용

크롤링 없는 실습

pip install geopy

In [ ]:
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Geocoding 클라이언트 초기화
# 반드시 user_agent를 설정해야 한다. (본인의 프로젝트명 등으로 설정)
# Nominatim 서비스를 사용하며, API Key는 필요 없다.
try:
    geolocator = Nominatim(user_agent="geocoding_analysis_tool")

    address = "부산광역시 부산진구 중앙대로 668"

    location = geolocator.geocode(address, timeout=10)

except Exception as e:
    print(f"API 요청 중 오류 발생: {e}")
    

latitude = location.latitude
longitude = location.longitude

In [77]:
latitude

35.1524287

In [78]:
longitude

129.0596192

pip install openmeteo-requests

pip install retry-requests

pip install pandas

pip install requests-cache

In [ ]:
import openmeteo_requests
import requests_cache
from retry_requests import retry
from datetime import datetime

# Open-Meteo 클라이언트 초기화
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# API 엔드포인트 URL
URL = "https://api.open-meteo.com/v1/forecast"

# 날씨 API 요청 (위경도 기반)

# 요청 매개변수 (Parameters) 설정
params = {
    "latitude": latitude,
    "longitude": longitude,
    "current": ["temperature_2m", "relative_humidity_2m", "weather_code", "wind_speed_10m"],
    "timezone": "Asia/Seoul",
    "forecast_days": 1
}

try:
    # API 호출을 실행한다.
    responses = openmeteo.weather_api(URL, params=params)
    response = responses[0]
    
    # 데이터 추출 및 구조화 (Current Weather)
    current = response.Current()

    current_data = {
        "시간": datetime.fromtimestamp(current.Time()).strftime('%Y-%m-%d %H:%M:%S'),
        "온도_2m": current.Variables(0).Value(),
        "상대_습도_2m": current.Variables(1).Value(),
        "날씨_코드": current.Variables(2).Value(),
        "풍속_10m": current.Variables(3).Value()
    }

except Exception as e:
    print(f"API 요청 중 오류 발생: {e}")

In [84]:
current_data

{'시간': '2025-12-04 17:15:00',
 '온도_2m': 3.25,
 '상대_습도_2m': 51.0,
 '날씨_코드': 0.0,
 '풍속_10m': 6.287129878997803}

In [46]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [49]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain how AI works in a few words in Korean",
)

print(response.text)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


**데이터를 학습해 인간처럼 생각하고 판단해요.**

(데이-터를 학습해 인간처럼 생각하고 판단해요.)

*Translation: It learns from data to think and judge like a human.*


In [48]:
prompt = f"""다음은 현재 날씨 정보 검색 결과입니다. 
이 결과를 분석하여 날씨 상황을 요약하고 설명해주세요.

**날씨 데이터:**
{current_data}
"""

print(prompt)

다음은 현재 날씨 정보 검색 결과입니다. 
이 결과를 분석하여 날씨 상황을 요약하고 설명해주세요.

**날씨 데이터:**
{'시간': '2025-12-04 17:15:00', '온도_2m': 3.25, '상대_습도_2m': 51.0, '날씨_코드': 0.0, '풍속_10m': 6.287129878997803}



In [50]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)
print(response.text)

제공된 날씨 데이터를 분석한 결과는 다음과 같습니다.

**날씨 상황 요약 및 설명:**

현재 날짜는 2025년 12월 4일 오후 5시 15분으로, 겨울철 초저녁 시간대의 날씨입니다.

*   **기온:** 3.25°C로 상당히 쌀쌀하거나 추운 편입니다. 영하권에 가까운 기온이므로 외출 시 따뜻한 옷차림이 필요합니다.
*   **습도:** 상대 습도는 51.0%로 보통 수준입니다. 건조하거나 습하지 않은 적당한 습도입니다.
*   **하늘 상태:** 날씨 코드 0.0은 대체로 **맑은 하늘**을 의미합니다. 구름 없이 깨끗한 날씨일 것으로 예상됩니다.
*   **바람:** 풍속은 6.29m/s로 **다소 강한 바람**이 불고 있습니다. 이 정도의 바람은 체감 온도를 더욱 낮게 만들어 실제 기온보다 훨씬 춥게 느껴지게 할 수 있습니다.

**종합적으로 볼 때,**
현재 날씨는 맑은 하늘 아래 기온은 3도 초반으로 쌀쌀하며, 다소 강한 바람이 불어 체감 온도는 더 낮을 것으로 예상되는 **추운 겨울 초저녁 날씨**입니다. 외출 시에는 두꺼운 외투와 방한용품으로 몸을 따뜻하게 유지하는 것이 중요합니다.


In [51]:
prompt = f"""다음은 현재 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 우산이 필요할지 알려주세요.

**날씨 데이터:**
```json
{current_data}
```
"""

print(prompt)

다음은 현재 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 우산이 필요할지 알려주세요.

**날씨 데이터:**
```json
{'시간': '2025-12-04 17:15:00', '온도_2m': 3.25, '상대_습도_2m': 51.0, '날씨_코드': 0.0, '풍속_10m': 6.287129878997803}
```



In [52]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)
print(response.text)

지금 외출하신다면 **우산은 필요하지 않을 것으로 보입니다.**

제공된 날씨 데이터의 **'날씨_코드'가 0.0**으로, 이는 **'맑은 하늘'**을 의미합니다. 현재 비나 눈이 오지 않는 상태입니다.


In [53]:
prompt = f"""다음은 현재 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요.

**날씨 데이터:**
```json
{current_data}
```
"""

print(prompt)

다음은 현재 날씨 데이터입니다. 이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요.

**날씨 데이터:**
```json
{'시간': '2025-12-04 17:15:00', '온도_2m': 3.25, '상대_습도_2m': 51.0, '날씨_코드': 0.0, '풍속_10m': 6.287129878997803}
```



In [54]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)
print(response.text)

현재 날씨 데이터를 분석한 결과, 지금 외출하신다면 **매우 춥게 느껴지실 것입니다.**

**날씨 요약:**
*   **시간:** 2025년 12월 4일 17시 15분 (초겨울 저녁 시간)
*   **기온:** 3.25°C (쌀쌀한 정도를 넘어 추움)
*   **풍속:** 6.29 m/s (약 22.6 km/h) (강한 바람)
*   **습도:** 51.0% (보통)
*   **날씨:** 맑음

**분석:**
기온은 3.25°C로 이미 쌀쌀하지만, 풍속이 6.29m/s (약 22.6km/h)로 강하게 불어 **체감 온도는 영하로 크게 떨어질 것으로 예상**됩니다. 맑은 날씨지만 바람 때문에 더욱 춥게 느껴질 겨울 저녁 날씨입니다.

**지금 외출하신다면 다음 드레스 코디를 추천합니다:**

따뜻하게 몸을 보호할 수 있는 **완벽한 겨울용 드레스 코드**를 추천합니다.

1.  **상의:**
    *   **이너웨어:** 내복 또는 히트텍 등 기능성 보온 이너웨어를 반드시 착용하세요.
    *   **미들웨어:** 그 위에 두꺼운 스웨터, 플리스, 또는 울 니트 등 보온성이 뛰어난 옷을 입으세요.
    *   **아우터:** **방풍 기능이 있는 두꺼운 패딩 점퍼나 다운 코트**를 입으세요. 바람을 완벽하게 막는 것이 가장 중요합니다.

2.  **하의:**
    *   **이너웨어:** 내복이나 기모 처리된 레깅스/타이즈를 안에 입으세요.
    *   **바지:** 그 위에 두꺼운 바지 (예: 기모 청바지, 울 바지, 두꺼운 면바지)를 착용하세요.

3.  **모자:**
    *   니트 모자나 비니 등 귀까지 덮는 따뜻한 모자를 착용하여 머리 쪽 체온 손실을 막으세요.

4.  **장갑:**
    *   손을 보호할 수 있는 두꺼운 장갑을 반드시 착용하세요.

5.  **목도리:**
    *   바람으로부터 목을 보호하고 체온을 유지할 수 있는 따뜻하고 두툼한 목도리를 둘러주세요.

6.  **신발:**
    *   방한 기능이 있는 부츠나

In [55]:
prompt = f"""다음은 현재 날씨 데이터입니다. 
이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요. 
결과는 현재 날씨에 어울리는 스타일로 HTML, CSS를 사용한 인포그래픽을 작성해주세요. HTML 문서 외 설명은 작성하지 마세요.

**날씨 데이터:**
```json
{current_data}
```
"""

print(prompt)

다음은 현재 날씨 데이터입니다. 
이 JSON 데이터를 분석하여 지금 외출을 한다면 어떤 드레스 코디를 하는 것이 좋을지 알려주세요. 
결과는 현재 날씨에 어울리는 스타일로 HTML, CSS를 사용한 인포그래픽을 작성해주세요. HTML 문서 외 설명은 작성하지 마세요.

**날씨 데이터:**
```json
{'시간': '2025-12-04 17:15:00', '온도_2m': 3.25, '상대_습도_2m': 51.0, '날씨_코드': 0.0, '풍속_10m': 6.287129878997803}
```



In [56]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
)
print(response.text)

```html
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>오늘의 드레스 코드</title>
    <style>
        /* General Styling */
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            display: flex;
            justify-content: center;
            align-items: center;
            min-height: 100vh;
            margin: 0;
            background: linear-gradient(to bottom right, #e0eaff, #f0f8ff); /* Light sky gradient */
            color: #333;
            padding: 20px;
            box-sizing: border-box;
        }

        .infographic-container {
            width: 90%;
            max-width: 850px;
            background-color: #ffffff;
            border-radius: 20px;
            box-shadow: 0 10px 30px rgba(0, 0, 0, 0.1);
            padding: 30px;
            display: flex;
            flex-direction: column;
            gap: 30px;
          

In [57]:
result = response.text
result = result.replace("```html","").replace("```","")
print(result)


<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>오늘의 드레스 코드</title>
    <style>
        /* General Styling */
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            display: flex;
            justify-content: center;
            align-items: center;
            min-height: 100vh;
            margin: 0;
            background: linear-gradient(to bottom right, #e0eaff, #f0f8ff); /* Light sky gradient */
            color: #333;
            padding: 20px;
            box-sizing: border-box;
        }

        .infographic-container {
            width: 90%;
            max-width: 850px;
            background-color: #ffffff;
            border-radius: 20px;
            box-shadow: 0 10px 30px rgba(0, 0, 0, 0.1);
            padding: 30px;
            display: flex;
            flex-direction: column;
            gap: 30px;
            borde

In [60]:
current_data

{'시간': '2025-12-04 17:15:00',
 '온도_2m': 3.25,
 '상대_습도_2m': 51.0,
 '날씨_코드': 0.0,
 '풍속_10m': 6.287129878997803}

In [67]:
current_data['시간'].split()[0].replace("-", "")

'20251204'

In [ ]:
# html 인포그래픽 생성
formatted_time = current_data['시간'].split()[0].replace("-", "")

file = open(f"result_{formatted_time}.html", "w", encoding="utf8")

file.write(str(result))
file.close()